# Inspect CCM lag screening
Keep the original climate processes fixed while comparing embedding span, random-library count, and target-shift range. This notebook checks directional evidence, not recovery of the equation's 20-year delay or a calibrated causal probability.

The screen follows the time-shift reasoning in [Ye et al. (2015)](https://doi.org/10.1038/srep14750) and [Van Nes et al. (2015)](https://doi.org/10.1038/nclimate2568), using native PyEDM library subsampling without Gaussian smoothing or the papers' full model-null tests.


In [ ]:
from pathlib import Path
from importlib.metadata import version
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits

from toolbox.process import generate_process
from toolbox.causal import run_ccm, filter_ccm_by_lag, summarize_ccm_lag

ROOT = Path.cwd()  # Run this notebook from the project root.
STEM = "02_2_inspect_ccm_lags"
DATA_DIR = ROOT / "data" / "processed" / STEM
FIGURE_DIR = ROOT / "figures" / STEM
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


## Set the scientific parameters
All physical times use kyr; BP ages decrease as the simulation advances. Library count, library size, and surrogate count are separate settings.


In [ ]:
SEED = 20261009                  # Original record used in experiment 02.
HOLDOUT_SEEDS = [20261301, 20261302]  # Two additional records; do not retune separately for them.
DURATION = 30.0                  # Total retained process duration, kyr.
DT = 0.020                       # Native uniform step, kyr: 20 years.
TRUE_LAG = 0.020                 # Direct X -> Y equation delay, kyr; no reverse coupling.
MEMORY_X = 0.5                   # X decay time, kyr.
MEMORY_Y = 0.2                   # Intrinsic Y decay time, kyr.
GAIN = 4.0                       # Long-run response to a held unit of standardized g(X).
BACKGROUND_X = 1 / np.sqrt(1 - 0.95**2)  # Stationary X SD in arbitrary units.
BACKGROUND_Y = 1 / np.sqrt(1 - 0.8**2)   # Unforced Y SD; not measurement noise.
BURN_TIMESCALES = 5              # Warm-up in units of the longest memory time.
ANALYSIS_OLD = 29.0               # Old edge of the common analysis interval, kyr BP.
ANALYSIS_YOUNG = 1.0              # Young edge; the interval contains 1401 native points.
PROCESS_NAMES = ["linear", "nonlinear", "independent"]  # Same innovations across process types.

ALPHA = 0.05                     # Original Tp=0 cut-and-swap upper-tail p threshold.
EMBEDDINGS = [(3, -5), (3, -1), (2, -1)]  # (E, tau in steps); spans are 200, 40, and 20 years.
EXCLUSION_RADIUS = 10            # Exclude neighbors within 200 years; fixed in all comparisons.
LIBRARY_FRACTION = 0.80           # Library size / number of valid embedded rows before the lag scan.
CCM_SAMPLES = 10                 # Random libraries averaged for each original Tp=0 score.
N_SURROGATES = 100               # Cut-and-swap null records per direction; p resolution is 1/101.
PILOT_LIBRARIES = 20             # Small library ensemble for the embedding comparison only.
LIBRARY_COUNTS = [20, 50, 100]    # Prefixes of the same larger ensemble; add 500 for a longer check.
MAX_LAG_STEPS = [5, 10, 15, 25]  # Symmetric search bounds: 100, 200, 300, and 500 years.
SIGN_THRESHOLD = 0.90            # Required stable peak-sign fraction; not a confidence level.

SELECTED_E = 3                   # Keep the existing embedding dimension after comparing candidates.
SELECTED_TAU = -1                # Shorten the embedding span to 40 years; validate below.
SELECTED_LAG_MAX = 15             # Production range +/-300 years, checked against the wider scan.
SELECTED_LIBRARIES = 100          # Production library repeats; 20 is used only for the initial pilot.


## Generate records and retain every lag curve
The generator matches experiment 02 and adds no observation or chronology errors. For a proposed X -> Y arrow, CCM reconstructs X(t + Tp) from Y's history; negative Tp means past X.


In [ ]:
def original_series(seed, process_name):
    ar_x = np.exp(-DT / MEMORY_X)
    ar_y = np.exp(-DT / MEMORY_Y)
    noise = (BACKGROUND_X * np.sqrt(1 - ar_x**2),
             BACKGROUND_Y * np.sqrt(1 - ar_y**2))
    gain = 0.0 if process_name == "independent" else GAIN
    form = "nonlinear" if process_name == "nonlinear" else "linear"
    process = generate_process(
        n_steps=int(round(DURATION / DT)) + 1, dt=DT, age_start=DURATION,
        lag=TRUE_LAG, ar_x=ar_x, ar_y=ar_y, coupling=(1 - ar_y) * gain,
        coupling_form=form, noise_std=noise,
        burn_in=int(np.ceil(BURN_TIMESCALES * max(MEMORY_X, MEMORY_Y) / DT)), rng=seed,
    )
    first = int(round((DURATION - ANALYSIS_OLD) / DT))
    last = int(round((DURATION - ANALYSIS_YOUNG) / DT))
    series = process["series"].iloc[first:last + 1].copy().reset_index(drop=True)
    series["age"] = np.linspace(ANALYSIS_OLD, ANALYSIS_YOUNG, len(series))
    series["time"] = ANALYSIS_OLD - series["age"]
    return series


def fit_case(series, seed, E, tau, n_libraries, max_lag, library_size=None):
    if library_size is None:
        library_size = int(LIBRARY_FRACTION * (len(series) - (E - 1) * abs(tau)))
    with threadpool_limits(limits=1):
        raw = run_ccm(
            series, E=E, tau=tau, exclusion_radius=EXCLUSION_RADIUS,
            lib_size=library_size, sample=CCM_SAMPLES, n_surrogates=N_SURROGATES,
            seed=seed + 300, alpha=ALPHA,
        )
        result = filter_ccm_by_lag(
            series, raw, lag_steps=np.arange(-max_lag, max_lag + 1),
            n_libraries=n_libraries, sign_threshold=SIGN_THRESHOLD, seed=seed + 300,
        )
    return result


def save_case(result, label):
    result["edges"].to_csv(DATA_DIR / f"{label}_edges.csv", index=False)
    arrays = {name: value for name, value in result["raw"].items() if isinstance(value, np.ndarray)}
    np.savez_compressed(DATA_DIR / f"{label}_raw.npz", **arrays)
    for name in ["lag_profile", "lag_peaks"]:
        result["raw"][name].to_csv(DATA_DIR / f"{label}_{name}.csv", index=False)
    (DATA_DIR / f"{label}_diagnostics.json").write_text(json.dumps(result["diagnostics"], indent=2))


def save_figure(fig, name):
    fig.savefig(FIGURE_DIR / f"{name}.png", dpi=180, bbox_inches="tight")
    fig.savefig(FIGURE_DIR / f"{name}.pdf", bbox_inches="tight")
    plt.show()


original = {name: original_series(SEED, name) for name in PROCESS_NAMES}
for name, series in original.items():
    series.to_csv(DATA_DIR / f"original_{name}.csv", index=False)
    print(name, "rows:", len(series), "age range:", series["age"].iloc[[0, -1]].tolist())


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
for ax, name in zip(axes, PROCESS_NAMES):
    series = original[name]
    ax.plot(series["age"], series["x"], label="X", linewidth=0.7)
    ax.plot(series["age"], series["y"], label="Y", linewidth=0.7, alpha=0.8)
    ax.set_title(f"{name.capitalize()}: " + ("no coupling" if name == "independent" else "X -> Y only"))
    ax.set_ylabel("Proxy value")
axes[0].legend(ncol=2)
axes[-1].set_xlim(ANALYSIS_OLD, ANALYSIS_YOUNG)
axes[-1].set_xlabel("Age (kyr BP)")
fig.tight_layout()
save_figure(fig, "01_original_records")


## Compare three embedding spans
Change only the CCM embedding, keeping records, time exclusion, library size, random seeds, and common anchors fixed. These 20 random libraries are a pilot; the selected configuration later uses the usual 80% library-size rule.


In [ ]:
embedding_results = {}
embedding_rows = []
pilot_history = max((E - 1) * abs(tau) for E, tau in EMBEDDINGS)
pilot_library_size = int(LIBRARY_FRACTION * (len(original["linear"]) - pilot_history))
print("Fixed embedding-pilot library size:", pilot_library_size)
for process_name in ["linear", "nonlinear"]:
    for E, tau in EMBEDDINGS:
        started = time.perf_counter()
        result = fit_case(original[process_name], SEED, E, tau, PILOT_LIBRARIES,
                          max(MAX_LAG_STEPS), library_size=pilot_library_size)
        embedding_results[(process_name, E, tau)] = result
        table = result["edges"].assign(process=process_name, E=E, tau=tau,
                                        embedding_span_years=(E - 1) * abs(tau) * DT * 1000)
        embedding_rows.append(table)
        save_case(result, f"embedding_{process_name}_E{E}_tau{tau}")
        print(process_name, E, tau, f"{time.perf_counter() - started:.1f} s", flush=True)
embedding_table = pd.concat(embedding_rows, ignore_index=True)
embedding_table.to_csv(DATA_DIR / "embedding_comparison.csv", index=False)
columns = ["process", "E", "tau", "source", "score", "p_value", "optimal_lag_kyr",
           "lag_past_fraction", "lag_future_fraction", "lag_status", "edge_detected"]
embedding_table[columns]


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharex=True, sharey=True)
for row, name in enumerate(["linear", "nonlinear"]):
    for column, (E, tau) in enumerate(EMBEDDINGS):
        result = embedding_results[(name, E, tau)]
        lags = result["raw"]["lag_steps"] * DT
        for direction, label, color in [(0, "X -> Y", "tab:blue"), (1, "Y -> X", "tab:orange")]:
            curves = result["raw"]["lag_rho"][:, :, direction]
            low, high = np.percentile(curves, [5, 95], axis=0)
            axes[row, column].plot(lags, curves.mean(axis=0), color=color, label=label)
            axes[row, column].fill_between(lags, low, high, color=color, alpha=0.15)
        axes[row, column].axvline(0, color="0.4", linestyle=":")
        axes[row, column].set_title(f"{name}: E={E}, tau={tau}; span={(E-1)*abs(tau)*DT*1000:.0f} yr")
        if row == 1:
            axes[row, column].set_xlabel("Target shift Tp (kyr)")
        if column == 0:
            axes[row, column].set_ylabel("Cross-map skill (rho)")
axes[0, 0].legend()
fig.suptitle("Embedding comparison: mean and 5-95% random-library range")
fig.tight_layout()
save_figure(fig, "02_embedding_comparison")


## Scan a wider window with more random libraries
The saved pilot retains the true arrow with all three embeddings; we keep E=3 and shorten tau to -1, moving the reverse peak farther from zero. The following scans keep this embedding fixed and use the standard 80% library rule (L=1119), while the embedding-only comparison above holds L=1112 fixed.


In [ ]:
wide_results = {}
for name in PROCESS_NAMES:
    started = time.perf_counter()
    result = fit_case(original[name], SEED, SELECTED_E, SELECTED_TAU,
                      max(LIBRARY_COUNTS), max(MAX_LAG_STEPS))
    wide_results[name] = result
    save_case(result, f"wide_{name}")
    print(name, f"{time.perf_counter() - started:.1f} s", flush=True)


## Compare library counts and search bounds
Reuse prefixes of the same library draws and subsets of the same wide scan, holding its anchors fixed. A peak at either search boundary is unresolved, and a nonsignificant Tp=0 result can never become a detected arrow.


In [ ]:
sensitivity_rows = []
for name, result in wide_results.items():
    for count in LIBRARY_COUNTS:
        for maximum in MAX_LAG_STEPS:
            selected = summarize_ccm_lag(
                result, n_libraries=count, lag_steps=np.arange(-maximum, maximum + 1),
                sign_threshold=SIGN_THRESHOLD,
            )
            table = selected["edges"].assign(process=name, n_libraries=count,
                                               max_shift_years=maximum * DT * 1000)
            sensitivity_rows.append(table)
sensitivity = pd.concat(sensitivity_rows, ignore_index=True)
sensitivity.to_csv(DATA_DIR / "count_window_sensitivity.csv", index=False)
sensitivity[["process", "source", "n_libraries", "max_shift_years", "optimal_lag_kyr",
             "lag_past_fraction", "lag_future_fraction", "lag_boundary_fraction", "lag_status", "edge_detected"]]


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(11, 10), sharex=True, sharey=True)
for row, name in enumerate(PROCESS_NAMES):
    for column, source in enumerate(["x", "y"]):
        ax = axes[row, column]
        subset = sensitivity[(sensitivity["process"] == name) & (sensitivity["source"] == source)]
        # Both signs are shown; unresolved peaks account for the remaining fraction.
        for count, color in zip(LIBRARY_COUNTS, ["0.6", "tab:green", "tab:blue", "tab:purple"]):
            group = subset[subset["n_libraries"] == count].sort_values("max_shift_years")
            ax.plot(group["max_shift_years"], group["lag_past_fraction"], "o-", color=color, label=f"B={count}, past")
            ax.plot(group["max_shift_years"], group["lag_future_fraction"], "s--", color=color, label=f"B={count}, future")
        ax.axhline(SIGN_THRESHOLD, color="0.2", linestyle=":", linewidth=1)
        ax.set_title(f"{name}: " + ("X -> Y" if source == "x" else "Y -> X"))
        ax.set_ylim(-0.03, 1.03)
        if column == 0:
            ax.set_ylabel("Fraction of random libraries")
        if row == 2:
            ax.set_xlabel("Maximum |Tp| (years)")
axes[0, 0].legend(fontsize=8, ncol=2)
fig.suptitle("Peak-sign stability versus random-library count and search bound")
fig.tight_layout()
save_figure(fig, "03_count_window_sensitivity")


## Check the selected window on separate records
Refit the selected window, including its actual common anchors, on the original record and two additional process realizations. Settings are fixed across all records; truth is used only to describe the resulting arrows.


In [ ]:
validation_results = {}
validation_rows = []
for seed in [SEED] + HOLDOUT_SEEDS:
    for name in PROCESS_NAMES:
        started = time.perf_counter()
        series = original_series(seed, name)
        result = fit_case(series, seed, SELECTED_E, SELECTED_TAU,
                          SELECTED_LIBRARIES, SELECTED_LAG_MAX)
        validation_results[(seed, name)] = result
        validation_rows.append(result["edges"].assign(process=name, process_seed=seed))
        save_case(result, f"validation_{seed}_{name}")
        print(seed, name, f"{time.perf_counter() - started:.1f} s", flush=True)
validation = pd.concat(validation_rows, ignore_index=True)
validation["true_edge"] = (validation["process"] != "independent") & (validation["source"] == "x")
validation.to_csv(DATA_DIR / "validation_results.csv", index=False)
validation[["process_seed", "process", "source", "true_edge", "score", "p_value",
            "raw_edge_detected", "edge_detected", "unoriented", "optimal_lag_kyr", "lag_status"]]


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(11, 9), sharex=True)
for row, name in enumerate(PROCESS_NAMES):
    result = validation_results[(SEED, name)]
    lags = result["raw"]["lag_steps"] * DT
    peaks = result["raw"]["lag_peaks"]
    for direction, source, label, color in [(0, "x", "X -> Y", "tab:blue"), (1, "y", "Y -> X", "tab:orange")]:
        curves = result["raw"]["lag_rho"][:, :, direction]
        low, high = np.percentile(curves, [5, 95], axis=0)
        axes[row, 0].plot(lags, curves.mean(axis=0), color=color, label=label)
        axes[row, 0].fill_between(lags, low, high, color=color, alpha=0.15)
        counts = peaks[peaks["source"] == source]["optimal_lag_kyr"].value_counts().sort_index()
        axes[row, 1].plot(counts.index, counts.values / SELECTED_LIBRARIES, "o-", color=color, label=label)
    for ax in axes[row]:
        ax.axvline(0, color="0.4", linestyle=":")
        ax.set_title(name.capitalize())
    axes[row, 0].set_ylabel("Cross-map skill (rho)")
    axes[row, 1].set_ylabel("Fraction of library optima")
axes[0, 0].legend()
axes[-1, 0].set_xlabel("Target shift Tp (kyr)")
axes[-1, 1].set_xlabel("Optimal target shift (kyr)")
fig.suptitle("Selected settings: curves and distributions of per-library optima")
fig.tight_layout()
save_figure(fig, "04_selected_lag_distributions")


## Save the settings and inspect the decision table
The original Tp=0 rho and p remain unchanged by the lag screen; peak rho is only a diagnostic. Stable sign support describes resampling of one record and does not establish a general false-positive rate.


In [ ]:
decisions = []
for (seed, name), group in validation.groupby(["process_seed", "process"], sort=False):
    forward = group[group["source"] == "x"].iloc[0]
    reverse = group[group["source"] == "y"].iloc[0]
    complete = group["edge_detected"].notna().all() and group["status"].eq("ok").all()
    unresolved = group["unoriented"].any()
    expected_forward = name != "independent"
    exact = complete and not unresolved and bool(forward["edge_detected"]) == expected_forward and not bool(reverse["edge_detected"])
    decisions.append({"seed": seed, "process": name, "complete": complete,
                      "x_to_y": forward["edge_detected"], "y_to_x": reverse["edge_detected"],
                      "any_unoriented": unresolved, "exact_graph": bool(exact)})
decision_table = pd.DataFrame(decisions)
decision_table.to_csv(DATA_DIR / "validation_decisions.csv", index=False)
settings = {name: value for name, value in globals().items()
            if name.isupper() and isinstance(value, (str, int, float, list))}
settings["packages"] = {name: version(name) for name in ["numpy", "pandas", "pyEDM", "scipy"]}
settings["interpretation"] = "Conditional random-library stability; original Tp=0 p; no causal-probability or lag-accuracy claim."
(DATA_DIR / "parameters.json").write_text(json.dumps(settings, indent=2))
decision_table


## Read the saved result
The selected settings are E=3, tau=-1, 100 random libraries, a +/-300-year search, and 90% peak-sign stability; +/-100 years truncates the coupled peaks, whereas the wider windows agree. Across the three seeds, all six coupled records retain only X -> Y and the three independent records retain no arrow; this small check does not calibrate a false-positive rate.


## Keep the scope explicit
This notebook fixes the model gain at K=4; the separate 03 experiment also tests weaker coupling. In its K=1 cases, an incorrect reverse arrow can have 96% negative-peak support, so random-library agreement is not a calibrated probability of causal correctness.
